# 03 - Farm Data Definitions Adapter

`bestpred-py` deliberately has no runtime dependency on `farm-data-definitions` (FDD).
It accepts the structural Cow/Herd attributes it needs and supplies temporary typed DTOs
for lactation facts that FDD does not yet model.

To use real FDD objects in a Bovi development checkout, install the sibling repository:

```bash
uv pip install -e ../farm-data-definitions
```

Or install the reviewed revision directly:

```bash
uv pip install "farm-data-definitions @ git+https://github.com/Bovi-analytics/farm-data-definitions.git@c964ece"
```

In [1]:
from pathlib import Path


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "packages/models/bestpred").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from a Bovi repository checkout")


ROOT = find_repo_root()
PACKAGE_ROOT = ROOT / "packages/models/bestpred"
FIXTURES = PACKAGE_ROOT / "tests/fixtures"
PARAMETERS = FIXTURES / "source11_current/bestpred.par"

## Create Cow and Herd identity objects

In [2]:
import os
from dataclasses import dataclass
from datetime import date
from uuid import UUID

fdd_enabled = os.getenv("BESTPRED_DISABLE_FDD") != "1"
try:
    if not fdd_enabled:
        raise ImportError("disabled for deterministic notebook validation")
    import farm_data_definitions as fdd

    herd_id = UUID("11111111-1111-1111-1111-111111111111")
    cow = fdd.Cow(
        animal_id=UUID("22222222-2222-2222-2222-222222222222"),
        herd_id=herd_id,
        gender=fdd.AnimalGender.FEMALE,
        birth_date=date(2020, 1, 1),
        animal_dhia="USA-DEMO-42",
        breed=[fdd.BreedPart(breed_code=fdd.Breed.HOL, proportion=1.0)],
    )
    herd = fdd.Herd(
        herd_id=herd_id,
        registration_number="DEMO01",
        state="35",
        country_code="US",
    )
    identity_mode = "real farm-data-definitions models"
except ImportError:

    @dataclass(frozen=True)
    class BreedPartDemo:
        breed_code: str
        proportion: float

    @dataclass(frozen=True)
    class CowDemo:
        animal_id: UUID
        birth_date: date
        breed: tuple[BreedPartDemo, ...]
        animal_dhia: str | None = None
        animal_usda: str | None = None
        animal_legal_id: str | None = None
        animal_ear_tag: str | None = None
        animal_farm_name: str | None = None

    @dataclass(frozen=True)
    class HerdDemo:
        herd_id: UUID
        state: str | None
        registration_number: str | None
        source_id: int | None = None

    cow = CowDemo(
        animal_id=UUID("22222222-2222-2222-2222-222222222222"),
        birth_date=date(2020, 1, 1),
        breed=(BreedPartDemo("HOL", 1.0),),
        animal_dhia="USA-DEMO-42",
    )
    herd = HerdDemo(
        herd_id=UUID("11111111-1111-1111-1111-111111111111"),
        state="35",
        registration_number="DEMO01",
    )
    identity_mode = "protocol-compatible demo objects (FDD is optional)"

print(f"Identity mode: {identity_mode}")

Identity mode: protocol-compatible demo objects (FDD is optional)


## Add the BESTPRED facts that FDD currently lacks

In [3]:
from bestpred.adapters.farm_data_definitions import (
    BestpredHerdMeansInput,
    BestpredLactationInput,
    BestpredTestDayInput,
    breed_code_from_cow,
    format4_record_from_fdd,
)

lactation = BestpredLactationInput(
    fresh_date=date(2024, 2, 3),
    parity=2,
    length=305,
    previous_days_open=140,
    herd_means=BestpredHerdMeansInput(milk=20_000, fat=700, protein=600, scs=280),
    test_days=tuple(
        BestpredTestDayInput(
            dim=dim,
            milk_yield=int(milk_lb * 10),
            fat_percent=int(fat_percent * 10),
            protein_percent=int(protein_percent * 10),
            scs=int(scs * 10),
        )
        for dim, milk_lb, fat_percent, protein_percent, scs in [
            (30, 70.0, 3.9, 3.2, 2.1),
            (60, 75.0, 3.8, 3.1, 2.2),
            (120, 67.0, 4.0, 3.3, 2.4),
            (200, 58.0, 4.1, 3.4, 2.6),
            (280, 46.0, 4.2, 3.5, 2.8),
        ]
    ),
)
record = format4_record_from_fdd(cow, herd, lactation)
print("Breed:", breed_code_from_cow(cow))
print("BESTPRED cow/herd:", record.cow_id, record.herd_id)
print("Sorted DIM:", [segment.dim for segment in record.segments])

Breed: HO
BESTPRED cow/herd: HUSADEMO42 35DEMO01
Sorted DIM: [30, 60, 120, 200, 280]


Unlike the DataFrame API, these temporary DTOs sit directly on the Format-4 boundary:
milk, percentages, and SCS are already scaled integers. The first two herd-id characters
must contain a numeric BESTPRED state code. Pass `state_code=` or `bestpred_herd_id=` when
`Herd.state` contains a name or abbreviation such as `NY`.

In [4]:
from bestpred import predict_records, prediction_from_dcr_row
from bestpred.io.parameters import read_parameters

parameters = read_parameters(PARAMETERS)
result_row = predict_records([record], parameters, source11_compat=False)[0]
prediction = prediction_from_dcr_row(result_row, test_id="fdd-demo-lactation")
prediction.model_dump()["milk"]

{'yield_305': 19372.6840875026,
 'yield_365': 21842.379559227724,
 'yield_lactation': 19372.6840875026,
 'yield_partial': 19372.6840875026,
 'persistency': 0.04227071619916384,
 'yield_reliability': 0.7502061168364239,
 'persistency_reliability': 0.8008944818491783,
 'expanded_yield': 19163.808587508247,
 'herd_305': 20000.0,
 'bumpiness': 0.0}

## What must move upstream before this becomes canonical FDD integration

In [5]:
import pandas as pd

gaps = pd.DataFrame(
    [
        ("Lactation", "Cow/herd reference, fresh date, parity, length, previous days open"),
        ("TestDay", "DIM, yields/components/SCS and collection metadata"),
        ("HerdProductionBaseline", "Unit-explicit 305-day milk/fat/protein/SCS means"),
        ("LactationPrediction", "Yield, DCR, reliability, persistency and provenance"),
    ],
    columns=["Missing FDD model", "Required content"],
)
gaps

,Missing FDD model,Required content
0,Lactation,"Cow/herd reference, fresh date, parity, length..."
1,TestDay,"DIM, yields/components/SCS and collection meta..."
2,HerdProductionBaseline,Unit-explicit 305-day milk/fat/protein/SCS means
3,LactationPrediction,"Yield, DCR, reliability, persistency and prove..."


Until those shared models exist, the adapter DTOs are an integration boundary rather
than a persistent ontology. Do not store them as a competing Bovi domain model.